# Sentence-level NER on HIPE-2020 (fr): 
- **GLiNER multilingual** (`urchade/gliner_multi-v2.1`): 209M, genuinely multilingual, zero-shot NER model. Has a **native** confidence score and supports batched inference, so it's much cheaper to run over the full dataset.

Pipeline:

1. Reconstruct sentences from the token-level extraction (`hipe2020_dev_fr_ocrqa.csv`), splitting on the `MISC` `EndOfSentence` flag and respecting `NoSpaceAfter` for detokenization.
2. Run GLiNER multilingual with the same PER/LOC/ORG/TIME/PROD labels, using its native confidence score.

In [ ]:
!pip install -q pandas transformers accelerate torch

In [1]:
import json
from typing import Optional

import pandas as pd
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

/Utilisateurs/pchau/.conda/envs/notebook/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Reconstruct sentences from the token-level extraction

Loads the token table produced by `hipe_ocr_ner_extraction.ipynb`. A sentence ends at the token whose `MISC` contains `EndOfSentence`; a token whose own `MISC` contains `NoSpaceAfter` means no space is inserted before the *next* token. Sentences never cross document boundaries.

In [2]:
CSV_PATH = "hipe2020_dev_fr_ocrqa.csv"
tokens_df = pd.read_csv(CSV_PATH, dtype={"token": str, "MISC": str})
tokens_df["MISC"] = tokens_df["MISC"].fillna("_")
print(tokens_df.shape)
tokens_df.head(3)

(37952, 12)


,document_id,token,NE-COARSE-LIT,MISC,ocr_word_known,entity_type,ocr_ok,entity_span_id,ocr_mean_mention,ocr_ratio_context_5,ocr_ratio_context_10,doc_ocr_mean
0,EXP-1888-08-03-a-i0030,FAITS,O,_,True,O,True,NaN,NaN,1.0,1.0,0.986667
1,EXP-1888-08-03-a-i0030,DIVERS,O,EndOfLine,True,O,True,NaN,NaN,1.0,1.0,0.986667
2,EXP-1888-08-03-a-i0030,La,O,_,True,O,True,NaN,NaN,1.0,1.0,0.986667


In [3]:
def reconstruct_sentences(df: pd.DataFrame) -> pd.DataFrame:
    """Group the token-level HIPE table into sentence-level text, using MISC EndOfSentence /
    NoSpaceAfter flags for detokenization. Returns one row per sentence with the row-index
    range of its tokens in `df`, so results can be joined back to per-token OCR/NER features."""
    sentences = []
    doc_id = None
    sent_idx = 0
    pieces: list[str] = []
    no_space_before_next = False
    row_start = None

    for i, row in df.iterrows():
        if row["document_id"] != doc_id:
            # New document: HIPE documents always end on an EndOfSentence token in practice,
            # so there should be no leftover buffer here.
            doc_id = row["document_id"]
            sent_idx = 0
            pieces = []
            no_space_before_next = False
            row_start = i

        if pieces and not no_space_before_next:
            pieces.append(" ")
        pieces.append(str(row["token"]))
        no_space_before_next = "NoSpaceAfter" in row["MISC"]

        if "EndOfSentence" in row["MISC"]:
            sentences.append(
                {
                    "document_id": doc_id,
                    "sentence_id": sent_idx,
                    "sentence_text": "".join(pieces),
                    "row_start": row_start,
                    "row_end": i,
                }
            )
            sent_idx += 1
            pieces = []
            no_space_before_next = False
            row_start = i + 1

    return pd.DataFrame(sentences)


sentences_df = reconstruct_sentences(tokens_df)
print(sentences_df.shape[0], "sentences across", sentences_df["document_id"].nunique(), "documents")
sentences_df.head(5)

1244 sentences across 43 documents


,document_id,sentence_id,sentence_text,row_start,row_end
0,EXP-1888-08-03-a-i0030,0,FAITS DIVERS La panique des éléphants au grand...,0,11
1,EXP-1888-08-03-a-i0030,1,"— Quatre dromadaires et huit éléphants, fourni...",12,51
2,EXP-1888-08-03-a-i0030,2,Les éléphants étaient enchaînés à la nuque ot ...,52,68
3,EXP-1888-08-03-a-i0030,3,Tout à coup les éléphants furent effrayés par ...,69,95
4,EXP-1888-08-03-a-i0030,4,"Six d'entre eux rompirent leurs chaînes, et, m...",96,130


## 2. GLiNER multilingual (`urchade/gliner_multi-v2.1`)

Added as a lighter, genuinely multilingual comparison point (209M params, French included). Unlike NuExtract3, GLiNER's `inference`/`predict_entities` returns a **native** per-entity `score` -- no logprob approximation needed here. It also supports true batched inference, so the full dataset runs in one pass instead of NuExtract3's one-`generate()`-per-sentence loop.

In [12]:
%pip install -q gliner tiktoken

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from gliner import GLiNER

GLINER_MODEL_ID = "urchade/gliner_multi-v2.1"
GLINER_LABELS = ["PERS", "LOC", "ORG","TIME","PROD"]  
GLINER_THRESHOLD = 0.5  # GLiNER's own default; lower to increase recall

gliner_model = GLiNER.from_pretrained(GLINER_MODEL_ID)

/Utilisateurs/pchau/.conda/envs/notebook/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 702.07it/s]


### Example: run on the same single sentence first

In [7]:
example_sentence = sentences_df.iloc[0]["sentence_text"]
print("Input sentence:", example_sentence)

Input sentence: FAITS DIVERS La panique des éléphants au grand cortège dc Munich.


In [8]:
gliner_example_entities = gliner_model.predict_entities(
    example_sentence, GLINER_LABELS, threshold=GLINER_THRESHOLD
)
print("Input sentence:", example_sentence)
print("\nEntities with native confidence score:")
for e in gliner_example_entities:
    print(e)

Input sentence: FAITS DIVERS La panique des éléphants au grand cortège dc Munich.

Entities with native confidence score:
{'start': 58, 'end': 64, 'text': 'Munich', 'label': 'LOC', 'score': 0.9551448225975037}


### Apply to all sentences (batched)

Much cheaper than NuExtract3, so a single batched call over the whole dataset is fine interactively (no checkpointing needed).

In [9]:
GLINER_BATCH_SIZE = 16
GLINER_RESULTS_PATH = "hipe2020_dev_fr_gliner_ner.csv"

all_sentences = sentences_df["sentence_text"].tolist()
gliner_predictions = gliner_model.inference(
    all_sentences, GLINER_LABELS, threshold=GLINER_THRESHOLD, batch_size=GLINER_BATCH_SIZE
)

gliner_records = []
for srow, ents in zip(sentences_df.to_dict("records"), gliner_predictions):
    if not ents:
        gliner_records.append(
            {
                "document_id": srow["document_id"],
                "sentence_id": srow["sentence_id"],
                "sentence_text": srow["sentence_text"],
                "entity_text": None,
                "entity_label": None,
                "confidence": None,
                "start": None,
                "end": None,
            }
        )
    else:
        for e in ents:
            gliner_records.append(
                {
                    "document_id": srow["document_id"],
                    "sentence_id": srow["sentence_id"],
                    "sentence_text": srow["sentence_text"],
                    "entity_text": e["text"],
                    "entity_label": e["label"],
                    "confidence": e["score"],
                    "start": e["start"],
                    "end": e["end"],
                }
            )

gliner_results_df = pd.DataFrame(gliner_records)
gliner_results_df.to_csv(GLINER_RESULTS_PATH, index=False)
print(gliner_results_df.shape, "-- saved to", GLINER_RESULTS_PATH)
gliner_results_df.head(20)

/Utilisateurs/pchau/.conda/envs/notebook/lib/python3.11/site-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 1141 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/Utilisateurs/pchau/.conda/envs/notebook/lib/python3.11/site-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 536 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/Utilisateurs/pchau/.conda/envs/notebook/lib/python3.11/site-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 692 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/Utilisateurs/pchau/.conda/envs/notebook/lib/python3.11/site-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 870 has been truncated to 384
  batch = 

(1608, 8) -- saved to hipe2020_dev_fr_gliner_ner.csv


,document_id,sentence_id,sentence_text,entity_text,entity_label,confidence,start,end
0,EXP-1888-08-03-a-i0030,0,FAITS DIVERS La panique des éléphants au grand...,Munich,LOC,0.955145,58.0,64.0
1,EXP-1888-08-03-a-i0030,1,"— Quatre dromadaires et huit éléphants, fourni...",Hambourg,LOC,0.961006,99.0,107.0
2,EXP-1888-08-03-a-i0030,1,"— Quatre dromadaires et huit éléphants, fourni...",Munich,LOC,0.961633,135.0,141.0
3,EXP-1888-08-03-a-i0030,2,Les éléphants étaient enchaînés à la nuque ot ...,NaN,NaN,NaN,NaN,NaN
4,EXP-1888-08-03-a-i0030,3,Tout à coup les éléphants furent effrayés par ...,NaN,NaN,NaN,NaN,NaN
5,EXP-1888-08-03-a-i0030,4,"Six d'entre eux rompirent leurs chaînes, et, m...",wigsstrasse,LOC,0.786980,126.0,137.0
6,EXP-1888-08-03-a-i0030,4,"Six d'entre eux rompirent leurs chaînes, et, m...",uno rue latérale,LOC,0.601375,160.0,176.0
7,EXP-1888-08-03-a-i0030,5,Ce fut uno indescriptible panique.,NaN,NaN,NaN,NaN,NaN
8,EXP-1888-08-03-a-i0030,6,Les spectateurs s'enfuyaient en désordre en po...,NaN,NaN,NaN,NaN,NaN
9,EXP-1888-08-03-a-i0030,7,Des chevaux terrifiés augmentaient le danger.,NaN,NaN,NaN,NaN,NaN


In [ ]:
gliner_results_df["entity_label"].unique()